# Stock Market Direction Prediction — Gradient Boosting

**Student ID:** IT25102034  
**Model:** Gradient Boosting Classifier  
**Task:** Predict whether the next closing price moves up (`1`) or down/not up (`0`).

Training and validation data are used for model selection. The test set is reserved for final evaluation.


## 1. Import libraries and configure the experiment

In [ ]:
from pathlib import Path
import json
import shutil
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             log_loss, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
OUTPUT_DIR = Path('IT25102034_Gradient_Boosting_Results')
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)
print('Setup completed.')


## 2. Upload the selected datasets

In [ ]:
required_files=['train_selected.csv','validation_selected.csv','test_selected.csv']
missing=[f for f in required_files if not Path(f).exists()]
if missing:
    try:
        from google.colab import files
        print('Upload:',', '.join(missing))
        files.upload()
    except ImportError:
        print('Place the CSV files beside this notebook.')
missing=[f for f in required_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}')
print('All required datasets are available.')

## 3. Load and validate data

In [ ]:
train_df=pd.read_csv('train_selected.csv')
val_df=pd.read_csv('validation_selected.csv')
test_df=pd.read_csv('test_selected.csv')
datasets={'Training':train_df,'Validation':val_df,'Testing':test_df}
expected=list(train_df.columns)
for name,df in datasets.items():
    assert list(df.columns)==expected, f'{name} columns do not match.'
    assert 'Target' in df.columns and df.isna().sum().sum()==0
    assert set(df.Target.unique()).issubset({0,1})
    print(f'{name:10s}: {len(df):,} rows, {df.shape[1]} columns')
display(train_df.head())

## 4. Target distribution and feature separation

In [ ]:
rows=[]
for name,df in datasets.items():
    c=df.Target.value_counts().sort_index()
    rows.append({'Dataset':name,'Down_or_Not_Up_0':int(c.get(0,0)),'Up_1':int(c.get(1,0)),'Up_Percentage':round(100*(df.Target==1).mean(),2)})
target_distribution=pd.DataFrame(rows)
display(target_distribution)
target_distribution.to_csv(OUTPUT_DIR/'target_distribution.csv',index=False)
ax=target_distribution.set_index('Dataset')[['Down_or_Not_Up_0','Up_1']].plot(kind='bar',figsize=(8,5),color=['#d95f5f','#4c9f70'])
ax.set(title='Target Distribution by Dataset',xlabel='Dataset',ylabel='Records')
ax.tick_params(axis='x',rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR/'01_target_distribution.png',dpi=200,bbox_inches='tight')
plt.show()
FEATURES=[c for c in train_df.columns if c!='Target']
X_train,y_train=train_df[FEATURES],train_df.Target
X_val,y_val=val_df[FEATURES],val_df.Target
X_test,y_test=test_df[FEATURES],test_df.Target
print('Features:',FEATURES)

## 5. Train a baseline Gradient Boosting model

Gradient Boosting builds decision trees sequentially. Each new tree focuses on correcting errors made by the current ensemble. This can learn non-linear relationships without requiring feature scaling.


In [ ]:
def metrics(y_true, y_pred, prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_true, prob)
    }

baseline_model = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE
)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_val)
baseline_prob = baseline_model.predict_proba(X_val)[:, 1]
baseline_metrics = metrics(y_val, baseline_pred, baseline_prob)
display(pd.DataFrame([baseline_metrics]).round(4))
print(classification_report(y_val, baseline_pred, target_names=['Down/Not Up', 'Up']))


## 6. Five-fold cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_model = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE
)
result = cross_validate(cv_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
cv_scores = pd.DataFrame({
    'Fold': range(1, 6),
    'Accuracy': result['test_accuracy'],
    'Precision': result['test_precision'],
    'Recall': result['test_recall'],
    'F1_Score': result['test_f1'],
    'ROC_AUC': result['test_roc_auc']
})
display(cv_scores.round(4))
display(cv_scores.drop(columns='Fold').agg(['mean', 'std']).round(4))
cv_scores.to_csv(OUTPUT_DIR / 'cross_validation_scores.csv', index=False)


## 7. Hyperparameter tuning

The number of boosting stages, learning rate, and tree depth are tuned to control model complexity. Only training data are used during the grid search.


In [ ]:
parameter_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [1, 2, 3]
}

grid = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    parameter_grid,
    scoring='f1',
    cv=cv,
    n_jobs=1,
    refit=True,
    return_train_score=True
)
grid.fit(X_train, y_train)
print('Best parameters:', grid.best_params_)
print('Best mean CV F1:', round(grid.best_score_, 4))
tuning_results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score')
tuning_results.to_csv(OUTPUT_DIR / 'hyperparameter_tuning_results.csv', index=False)
display(tuning_results[['params', 'mean_train_score', 'mean_test_score',
                        'std_test_score', 'rank_test_score']].head(12))


## 8. Compare baseline and tuned models on validation data

In [ ]:
tuned_model=grid.best_estimator_
tuned_pred=tuned_model.predict(X_val)
tuned_prob=tuned_model.predict_proba(X_val)[:,1]
tuned_metrics=metrics(y_val,tuned_pred,tuned_prob)
comparison=pd.DataFrame([{'Model':'Baseline Gradient Boosting',**baseline_metrics},{'Model':'Tuned Gradient Boosting',**tuned_metrics}])
display(comparison.round(4))
comparison.to_csv(OUTPUT_DIR/'validation_model_comparison.csv',index=False)
ax=comparison.set_index('Model').plot(kind='bar',figsize=(10,5),ylim=(0,1),colormap='autumn')
ax.set(title='Baseline vs Tuned Gradient Boosting — Validation Set',ylabel='Score')
ax.tick_params(axis='x',rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_DIR/'02_validation_model_comparison.png',dpi=200,bbox_inches='tight')
plt.show()

## 9. Train final model and evaluate the test set

In [ ]:
X_final = pd.concat([X_train, X_val], ignore_index=True)
y_final = pd.concat([y_train, y_val], ignore_index=True)
final_model = GradientBoostingClassifier(**grid.best_params_, random_state=RANDOM_STATE)
final_model.fit(X_final, y_final)
test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = metrics(y_test, test_pred, test_prob)
display(pd.DataFrame([test_metrics]).round(4))
print(classification_report(y_test, test_pred, target_names=['Down/Not Up', 'Up']))
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / 'final_test_metrics.csv', index=False)
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(classification_report(y_test, test_pred, target_names=['Down/Not Up', 'Up']))


## 10. Evaluation curves, staged validation loss, and feature importance


In [ ]:
cm = confusion_matrix(y_test, test_pred)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr', cbar=False,
            xticklabels=['Down/Not Up', 'Up'], yticklabels=['Down/Not Up', 'Up'], ax=axes[0])
axes[0].set(title='Confusion Matrix', xlabel='Predicted class', ylabel='Actual class')

fpr, tpr, _ = roc_curve(y_test, test_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['ROC_AUC']:.3f}")
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set(title='ROC Curve', xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[1].legend()

pp, pr, _ = precision_recall_curve(y_test, test_prob)
axes[2].plot(pr, pp, color='#ea580c')
axes[2].set(title='Precision–Recall Curve', xlabel='Recall', ylabel='Precision')
plt.tight_layout()
plt.savefig(PLOT_DIR / '03_final_evaluation_curves.png', dpi=200, bbox_inches='tight')
plt.show()

staged_losses = [log_loss(y_val, p) for p in tuned_model.staged_predict_proba(X_val)]
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(staged_losses) + 1), staged_losses, color='#b45309')
plt.xlabel('Boosting stage')
plt.ylabel('Validation log loss')
plt.title('Gradient Boosting Validation Loss by Stage')
plt.tight_layout()
plt.savefig(PLOT_DIR / '04_staged_validation_loss.png', dpi=200, bbox_inches='tight')
plt.show()

importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': final_model.feature_importances_
}).sort_values('Importance', ascending=False)
display(importance)
importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
plt.figure(figsize=(8, 5))
sns.barplot(data=importance, x='Importance', y='Feature', color='#d97706')
plt.title('Gradient Boosting Feature Importance')
plt.tight_layout()
plt.savefig(PLOT_DIR / '05_feature_importance.png', dpi=200, bbox_inches='tight')
plt.show()


## 11. Save predictions, model and experiment summary

In [ ]:
predictions = X_test.copy()
predictions['Actual_Target'] = y_test.to_numpy()
predictions['Predicted_Target'] = test_pred
predictions['Probability_Up'] = test_prob
predictions['Prediction_Correct'] = predictions.Actual_Target == predictions.Predicted_Target
predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
display(predictions.head(15))

joblib.dump(final_model, OUTPUT_DIR / 'IT25102034_gradient_boosting_model.joblib')
summary = {
    'student_id': 'IT25102034',
    'algorithm': 'Gradient Boosting Classifier',
    'problem_type': 'Binary classification',
    'target_definition': '1 if next closing price is higher; otherwise 0',
    'features': FEATURES,
    'training_rows': int(len(train_df)),
    'validation_rows': int(len(val_df)),
    'testing_rows': int(len(test_df)),
    'best_parameters': grid.best_params_,
    'best_cross_validation_f1': float(grid.best_score_),
    'baseline_validation_metrics': {k: float(v) for k, v in baseline_metrics.items()},
    'tuned_validation_metrics': {k: float(v) for k, v in tuned_metrics.items()},
    'final_test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'limitations': [
        'Boosting is less interpretable than a single Decision Tree.',
        'Sequential training can be slower than parallel ensemble methods.',
        'Historical patterns do not guarantee future performance.',
        'Selected features omit news and macroeconomic variables.'
    ]
}
with open(OUTPUT_DIR / 'experiment_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


## 12. Conclusion

Report final Accuracy, Precision, Recall, F1 and ROC-AUC. Compare baseline and tuned validation results, explain sequential error correction and the learning-rate/tree-count trade-off, and discuss performance against the other five models and the majority-class baseline.


## 13. Create and download the results ZIP

In [ ]:
zip_path = shutil.make_archive('IT25102034_Gradient_Boosting_Results', 'zip', OUTPUT_DIR)
print('Created:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Download the ZIP from the notebook folder:', zip_path)
